# Re-tuning the semantic ↔ keyword balance

Production fuses a semantic ranking with a BM25 keyword ranking. How much each side counts is the single most important retrieval knob, and the current value has **never been fitted for the system as it exists today**.

`RRF_W = 0.85` was chosen in Phase 6 under conditions that have all since changed:

| fitted on | today |
|---|---|
| 739 **clause** chunks | 531 **whole articles** (Phase 12a) |
| eval v1, 40 questions, 8 cross-lingual | eval v3, 60 questions, 0 cross-lingual |
| statute-worded questions | naturally-worded questions (Phase 14) |
| scored at `top_k=10` | **`top_k=5`** (Phase 17) |

That last one matters more than it looks: Phase 16's sweep found a peak near `w = 0.80`, but it scored the top 10. Production now returns 5, so the weight has to be optimised for the list the generator actually receives.

## Two different parameters, both called "the balance"

Phase 6 tested both and this notebook re-tests both, because the project's own history uses "beta" and "w" for different things:

- **RRF weight `w`** — fuse by *rank*: `score = w/(k+rank_sem) + (1−w)/(k+rank_kw)`. Ignores how confident each ranker was.
- **Weighted-sum `beta`** — fuse by *score*: `score = beta·sem_norm + (1−beta)·kw_norm`, after normalising each side per query. Keeps confidence information that RRF discards.

RRF won in Phase 6. Whether it still wins on whole articles with natural questions is an open question, not a settled one.

## The trap this notebook is built around

Sweeping a parameter on 60 questions will *always* produce a winner. The Phase 6 values are the cautionary tale: they were fitted honestly and still failed to generalise, because the argmax of a noisy curve is not a stable quantity.

So the deliverable here is **not** the best value. It is:

1. the shape of the curve (is the peak real, or is the region flat?), and
2. a **bootstrap over questions** — resample the 60 with replacement 2,000 times, re-find the optimum each time, and look at the spread. If the optimum jumps around, the honest recommendation is a *region*, not a number.

## Cost: zero

All corpus and query vectors are cached from Phases 13/15; BM25 is local. No API calls.

Both candidate embedding models are swept, since the Gemini-vs-Cohere decision is still open and the best weight may differ between them.

In [1]:
import sys
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi

PROJECT = Path(r"c:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project")
CACHE = PROJECT / "eval" / "emb_cache"
sys.path.insert(0, str(PROJECT))
from src.retrieval import tokenize, RRF_K, RRF_W, BM25_K1, BM25_B, CANDIDATE_TOP_K

chunks = pd.read_parquet(PROJECT / "0. cleaned_data" / "chunks_articles.parquet")
chunks["article_number"] = chunks["article_number"].astype(str)
ev = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_v3.jsonl", encoding="utf-8")])

CH = hashlib.md5("".join(chunks["chunk_id"] + "\x00" + chunks["text"]).encode()).hexdigest()[:12]
QH = hashlib.md5("".join(ev["question_id"] + "\x00" + ev["question_text"]).encode()).hexdigest()[:12]

PROD_TOP_K = 5          # Phase 17 production default -- tune for the list actually returned
N = len(chunks)

bm25 = BM25Okapi([tokenize(t) for t in chunks["text"]], k1=BM25_K1, b=BM25_B)
BM_SCORES = np.vstack([bm25.get_scores(tokenize(q)) for q in ev["question_text"]])

# gold[qi] = row indices of every article that correctly answers question qi
GOLD = []
for q in ev.itertuples(index=False):
    want = {str(a) for a in q.answer_article_numbers}
    GOLD.append({i for i, c in enumerate(chunks.itertuples())
                 if c.law == q.answer_law and c.language == q.answer_language
                 and c.article_number in want})

MODELS = ["gemini-embedding-001", "cohere-embed-v4"]
SIMS = {}
for m in MODELS:
    co = torch.from_numpy(np.load(CACHE / f"{m}__{CH}__corpus.npy"))
    qu = torch.from_numpy(np.load(CACHE / f"{m}__{QH}__query.npy"))
    SIMS[m] = (qu @ co.T).numpy()

print(f"current production: RRF_K={RRF_K}, RRF_W={RRF_W}, top_k={PROD_TOP_K}")
print(f"{len(ev)} questions x {N} articles | models: {MODELS} | no API calls")

current production: RRF_K=5, RRF_W=0.85, top_k=5
60 questions x 531 articles | models: ['gemini-embedding-001', 'cohere-embed-v4'] | no API calls


## Scoring machinery

`rank_of` returns, for each question, the position of the first correct article in a ranking — or `None` if it is not in the returned list. Everything below is built on that.

Both fusion methods are implemented over **all 531 articles** rather than a candidate cutoff, so the two are compared on equal terms and no result is an artifact of `CANDIDATE_TOP_K`.

In [2]:
def ranks_from_scores(fused, top_k):
    """fused: (n_questions, n_docs). Returns array of 1-indexed ranks, NaN if not in top_k."""
    idx = np.argpartition(-fused, kth=min(top_k, fused.shape[1] - 1), axis=1)[:, :top_k]
    out = np.full(len(fused), np.nan)
    for qi in range(len(fused)):
        order = idx[qi][np.argsort(-fused[qi, idx[qi]])]
        for pos, d in enumerate(order, start=1):
            if d in GOLD[qi]:
                out[qi] = pos
                break
    return out


def mrr(r):
    return np.nansum(1.0 / r[~np.isnan(r)]) / len(r)


def recall_at(r, k):
    return np.sum(~np.isnan(r) & (r <= k)) / len(r)


# --- RRF: fuse by rank -------------------------------------------------------
def rank_matrix(scores):
    """Dense 1-indexed rank of every doc, per question (rank 1 = best)."""
    order = np.argsort(-scores, axis=1)
    rk = np.empty_like(order)
    rows = np.arange(scores.shape[0])[:, None]
    rk[rows, order] = np.arange(1, scores.shape[1] + 1)[None, :]
    return rk


def rrf_fused(sem_scores, kw_scores, w, k):
    return w / (k + rank_matrix(sem_scores)) + (1 - w) / (k + rank_matrix(kw_scores))


# --- weighted sum: fuse by score, normalised per query -----------------------
def minmax(x):
    lo = x.min(axis=1, keepdims=True)
    hi = x.max(axis=1, keepdims=True)
    return (x - lo) / np.maximum(hi - lo, 1e-12)


def wsum_fused(sem_scores, kw_scores, beta):
    return beta * minmax(sem_scores) + (1 - beta) * minmax(kw_scores)


# sanity: at the production setting this must reproduce the Phase 16 hybrid numbers
_r = ranks_from_scores(rrf_fused(SIMS["cohere-embed-v4"], BM_SCORES, RRF_W, RRF_K), 10)
print(f"sanity check -- cohere @ w={RRF_W}, k={RRF_K}, top_k=10: "
      f"recall@1 {recall_at(_r, 1):.3f}, MRR {mrr(_r):.3f}   (Phase 16 saw 0.733 / 0.838)")

sanity check -- cohere @ w=0.85, k=5, top_k=10: recall@1 0.733, MRR 0.838   (Phase 16 saw 0.733 / 0.838)


## 1. Fine sweep of the RRF weight, at production's `top_k=5`

In [3]:
# step 0.01, not 0.02 -- a coarser grid silently excludes the current production value
# of 0.85 and every lookup against it comes back empty.
WS = np.round(np.arange(0.0, 1.001, 0.01), 3)
assert np.isclose(WS, RRF_W).any(), "sweep grid must contain the current production weight"
rows = []
for m in MODELS:
    for w in WS:
        r = ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, w, RRF_K), PROD_TOP_K)
        rows.append({"model": m, "method": "rrf", "param": w,
                     "recall@1": recall_at(r, 1), "recall@5": recall_at(r, 5), "mrr": mrr(r)})
rrf_sweep = pd.DataFrame(rows)

print(f"RRF weight sweep at top_k={PROD_TOP_K}  (w=0 pure BM25, w=1 pure semantic)")
piv = rrf_sweep.pivot(index="param", columns="model", values="mrr")
print(piv.iloc[::5].round(3).to_string())

print("\nargmax per model:")
for m in MODELS:
    s = rrf_sweep[rrf_sweep.model == m]
    b = s.loc[s.mrr.idxmax()]
    cur = s[np.isclose(s.param, RRF_W)].iloc[0]
    print(f"  {m:24s} best w={b.param:.2f} MRR {b.mrr:.4f}  |  "
          f"current w={RRF_W} MRR {cur.mrr:.4f}  (gain {b.mrr - cur.mrr:+.4f})")

    # how wide is the region that is statistically indistinguishable from the peak?
    near = s[s.mrr >= b.mrr - 0.005]
    print(f"{'':26s} within 0.005 of peak: w in [{near.param.min():.2f}, {near.param.max():.2f}]"
          f"  ({len(near)} of {len(s)} settings)")

RRF weight sweep at top_k=5  (w=0 pure BM25, w=1 pure semantic)
model  cohere-embed-v4  gemini-embedding-001
param                                       
0.00             0.429                 0.429
0.05             0.429                 0.429
0.10             0.433                 0.433
0.15             0.447                 0.457
0.20             0.487                 0.480
0.25             0.510                 0.544
0.30             0.530                 0.576
0.35             0.564                 0.607
0.40             0.624                 0.679
0.45             0.663                 0.722
0.50             0.726                 0.774
0.55             0.749                 0.827
0.60             0.753                 0.871
0.65             0.794                 0.868
0.70             0.811                 0.905
0.75             0.843                 0.913
0.80             0.846                 0.931
0.85             0.838                 0.931
0.90             0.829              

## 2. Does the RRF `k` constant matter?

`k = 5` came from Phase 6. The canonical RRF paper uses 60. Small `k` sharpens the top of each list; large `k` flattens it.

In [4]:
KS = [1, 2, 3, 5, 10, 20, 40, 60]
rows = []
for m in MODELS:
    for k in KS:
        for w in np.round(np.arange(0.3, 1.001, 0.05), 3):
            r = ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, w, k), PROD_TOP_K)
            rows.append({"model": m, "k": k, "w": w, "mrr": mrr(r), "recall@1": recall_at(r, 1)})
kw_grid = pd.DataFrame(rows)

for m in MODELS:
    g = kw_grid[kw_grid.model == m]
    print(f"--- {m}: best MRR at each k")
    best = g.loc[g.groupby("k")["mrr"].idxmax()][["k", "w", "recall@1", "mrr"]]
    print(best.round(4).to_string(index=False))
    top = g.loc[g.mrr.idxmax()]
    print(f"    overall best: k={int(top.k)}, w={top.w:.2f}, MRR {top.mrr:.4f}\n")

--- gemini-embedding-001: best MRR at each k
 k    w  recall@1    mrr
 1 0.70    0.8833 0.9264
 2 0.70    0.8833 0.9297
 3 0.80    0.8833 0.9306
 5 0.80    0.8833 0.9306
10 0.85    0.8833 0.9306
20 0.90    0.9000 0.9389
40 0.95    0.9000 0.9389
60 0.95    0.9000 0.9389
    overall best: k=20, w=0.90, MRR 0.9389

--- cohere-embed-v4: best MRR at each k
 k    w  recall@1    mrr
 1 0.80    0.7167 0.8292
 2 0.70    0.7333 0.8297
 3 0.75    0.7500 0.8456
 5 0.80    0.7500 0.8464
10 0.85    0.7667 0.8519
20 0.90    0.7667 0.8547
40 0.95    0.7833 0.8603
60 0.95    0.8000 0.8700
    overall best: k=60, w=0.95, MRR 0.8700



## 3. Weighted-sum `beta` — the other kind of balance

Fuses normalised *scores* rather than ranks, so it keeps the confidence information RRF throws away. Phase 6 found RRF better; re-testing on today's setup.

In [5]:
rows = []
for m in MODELS:
    for b in np.round(np.arange(0.0, 1.001, 0.01), 3):
        r = ranks_from_scores(wsum_fused(SIMS[m], BM_SCORES, b), PROD_TOP_K)
        rows.append({"model": m, "method": "wsum", "param": b,
                     "recall@1": recall_at(r, 1), "recall@5": recall_at(r, 5), "mrr": mrr(r)})
wsum_sweep = pd.DataFrame(rows)

print(f"weighted-sum beta sweep at top_k={PROD_TOP_K}  (beta=0 pure BM25, beta=1 pure semantic)")
print(wsum_sweep.pivot(index="param", columns="model", values="mrr").iloc[::5].round(3).to_string())

print("\nbest of each method, side by side:")
allsw = pd.concat([rrf_sweep, wsum_sweep])
for m in MODELS:
    for meth in ["rrf", "wsum"]:
        s = allsw[(allsw.model == m) & (allsw.method == meth)]
        b = s.loc[s.mrr.idxmax()]
        print(f"  {m:24s} {meth:5s} best param {b.param:.2f}  recall@1 {b['recall@1']:.3f}  MRR {b.mrr:.4f}")
allsw.to_csv(PROJECT / "eval" / "fusion_weight_sweep.csv", index=False, encoding="utf-8-sig")

weighted-sum beta sweep at top_k=5  (beta=0 pure BM25, beta=1 pure semantic)


model  cohere-embed-v4  gemini-embedding-001
param                                       
0.00             0.429                 0.429
0.05             0.440                 0.433
0.10             0.455                 0.470
0.15             0.469                 0.498
0.20             0.479                 0.521
0.25             0.528                 0.554
0.30             0.555                 0.575
0.35             0.565                 0.636
0.40             0.625                 0.681
0.45             0.656                 0.713
0.50             0.712                 0.756
0.55             0.734                 0.792
0.60             0.749                 0.821
0.65             0.789                 0.846
0.70             0.800                 0.871
0.75             0.821                 0.910
0.80             0.834                 0.926
0.85             0.853                 0.943
0.90             0.880                 0.943
0.95             0.875                 0.947
1.00      

## 4. Is the optimum real, or is it noise?

This is the part that decides whether any number here should be adopted.

The 60 questions are resampled with replacement 2,000 times. On each resample the whole sweep is re-run and the argmax recorded. If the peak is a genuine property of the task, the optimum will cluster tightly. If it is an artifact of which 60 questions happen to be in the set, it will scatter — and the correct output is a plateau, not a point.

This is exactly the check Phase 6 lacked, and its absence is the likeliest reason those values did not survive contact with a changed corpus.

In [6]:
rng = np.random.default_rng(0)
B = 2000
boot = {}
for m in MODELS:
    # precompute rank arrays once per w, then resample question indices -- cheap
    per_w = {}
    for w in WS:
        per_w[w] = ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, w, RRF_K), PROD_TOP_K)
    R = np.vstack([per_w[w] for w in WS])          # (n_w, n_questions)
    inv = np.where(np.isnan(R), 0.0, 1.0 / np.nan_to_num(R, nan=1.0))

    picks = []
    for _ in range(B):
        idx = rng.integers(0, len(ev), len(ev))
        picks.append(WS[np.argmax(inv[:, idx].mean(axis=1))])
    boot[m] = np.array(picks)

print(f"bootstrap over questions, B={B}, RRF weight")
for m in MODELS:
    p = boot[m]
    lo, hi = np.percentile(p, [5, 95])
    print(f"\n  {m}")
    print(f"    median optimal w : {np.median(p):.2f}")
    print(f"    90% interval     : [{lo:.2f}, {hi:.2f}]   <- width {hi - lo:.2f}")
    print(f"    fraction of resamples choosing w >= 0.8 : {(p >= 0.8).mean():.2f}")
    print(f"    fraction choosing the current w = {RRF_W}   : {np.isclose(p, RRF_W).mean():.3f}")

bootstrap over questions, B=2000, RRF weight

  gemini-embedding-001
    median optimal w : 0.79
    90% interval     : [0.73, 0.79]   <- width 0.06
    fraction of resamples choosing w >= 0.8 : 0.00
    fraction choosing the current w = 0.85   : 0.000

  cohere-embed-v4
    median optimal w : 0.82
    90% interval     : [0.75, 0.83]   <- width 0.08
    fraction of resamples choosing w >= 0.8 : 0.56
    fraction choosing the current w = 0.85   : 0.000


In [7]:
# What would each candidate weight actually cost, on the real 60 questions?
print(f"MRR at candidate settings (RRF, k={RRF_K}, top_k={PROD_TOP_K})\n")
cands = [0.70, 0.75, 0.80, 0.85, 0.90, 1.00]


def at(sweep_df, model, w, col="mrr"):
    """Float-safe lookup -- never index a swept grid with a raw float literal."""
    s = sweep_df[(sweep_df.model == model) & np.isclose(sweep_df.param, w)]
    assert len(s) == 1, f"w={w} not on the sweep grid for {model}"
    return float(s.iloc[0][col])


tbl = []
for m in MODELS:
    tbl.append({"model": m, **{f"w={w:.2f}": round(at(rrf_sweep, m, w), 4) for w in cands}})
print(pd.DataFrame(tbl).to_string(index=False))

print("\nspread between best and worst of these candidates:")
for m in MODELS:
    vals = np.array([at(rrf_sweep, m, w) for w in cands])
    print(f"  {m:24s} {vals.max() - vals.min():.4f} MRR  "
          f"(~{(vals.max() - vals.min()) * len(ev):.1f} questions' worth)")

MRR at candidate settings (RRF, k=5, top_k=5)

               model  w=0.70  w=0.75  w=0.80  w=0.85  w=0.90  w=1.00
gemini-embedding-001  0.9047  0.9131  0.9306  0.9306  0.9222  0.9222
     cohere-embed-v4  0.8114  0.8428  0.8464  0.8375  0.8292  0.8278

spread between best and worst of these candidates:
  gemini-embedding-001     0.0258 MRR  (~1.5 questions' worth)
  cohere-embed-v4          0.0350 MRR  (~2.1 questions' worth)


## 5. The honest comparison: split-half cross-validation

Sections 1–4 tuned every method on all 60 questions and then reported its best score. That number is **optimistically biased for every method**, and worse, the bias grows with how many parameters a method has — so comparing "best RRF" against "best weighted-sum" against "best (k, w) grid" on those figures would systematically favour whichever method has the most knobs to overfit with.

The fix is to never score a parameter on the data that chose it. Each of 500 repetitions:

1. splits the 60 questions at random into 30 *tune* and 30 *test*,
2. picks each method's best parameter on the **tune** half only,
3. scores that parameter on the **test** half.

The production setting (RRF, `k=5`, `w=0.85`) is scored on the same test halves as a fixed baseline. It tunes nothing, so it carries no optimism — which makes it the fair thing to beat.

Three configurations compete, deliberately including one with an extra degree of freedom so the cost of that freedom is visible:

- **RRF (w only)**, `k = 5` fixed — one parameter
- **RRF (k, w)** — two parameters
- **weighted sum (beta)** — one parameter

In [8]:
BS = np.round(np.arange(0.0, 1.001, 0.01), 3)
KS_CV = [1, 2, 3, 5, 10, 20, 40, 60]
rng2 = np.random.default_rng(7)
N_SPLIT = 500


def recip(ranks):
    return np.where(np.isnan(ranks), 0.0, 1.0 / np.nan_to_num(ranks, nan=1.0))


# Precompute reciprocal-rank vectors once per configuration; CV is then pure indexing.
CONFIGS = {}
for m in MODELS:
    rrf_w = np.vstack([recip(ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, w, RRF_K), PROD_TOP_K))
                       for w in WS])
    rrf_kw, kw_labels = [], []
    for k in KS_CV:
        for w in WS[::5]:                      # coarser w grid so the (k,w) space stays comparable
            rrf_kw.append(recip(ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, w, k), PROD_TOP_K)))
            kw_labels.append((k, w))
    ws_b = np.vstack([recip(ranks_from_scores(wsum_fused(SIMS[m], BM_SCORES, b), PROD_TOP_K))
                      for b in BS])
    CONFIGS[m] = {
        "RRF (w only, k=5)": (rrf_w, [float(w) for w in WS]),
        "RRF (k and w)":     (np.vstack(rrf_kw), kw_labels),
        "weighted sum (beta)": (ws_b, [float(b) for b in BS]),
    }

# fixed, untuned baseline
BASE = {m: recip(ranks_from_scores(rrf_fused(SIMS[m], BM_SCORES, RRF_W, RRF_K), PROD_TOP_K))
        for m in MODELS}

results = []
for m in MODELS:
    acc = {name: [] for name in CONFIGS[m]}
    acc["production (w=0.85, k=5)"] = []
    chosen = {name: [] for name in CONFIGS[m]}
    for _ in range(N_SPLIT):
        perm = rng2.permutation(len(ev))
        tr, te = perm[:len(ev) // 2], perm[len(ev) // 2:]
        for name, (mat, labels) in CONFIGS[m].items():
            best = int(np.argmax(mat[:, tr].mean(axis=1)))
            acc[name].append(mat[best, te].mean())
            chosen[name].append(labels[best])
        acc["production (w=0.85, k=5)"].append(BASE[m][te].mean())
    for name, vals in acc.items():
        v = np.array(vals)
        results.append({"model": m, "config": name, "test_mrr_mean": v.mean(),
                        "test_mrr_sd": v.std()})

cv = pd.DataFrame(results)
print(f"OUT-OF-SAMPLE test MRR, {N_SPLIT} random 30/30 splits, top_k={PROD_TOP_K}")
print("(parameter chosen on the tune half, scored on the untouched test half)\n")
for m in MODELS:
    print(f"--- {m}")
    sub = cv[cv.model == m].sort_values("test_mrr_mean", ascending=False)
    print(sub[["config", "test_mrr_mean", "test_mrr_sd"]].round(4).to_string(index=False))
    print()
cv.to_csv(PROJECT / "eval" / "fusion_weight_cv.csv", index=False, encoding="utf-8-sig")

OUT-OF-SAMPLE test MRR, 500 random 30/30 splits, top_k=5
(parameter chosen on the tune half, scored on the untouched test half)

--- gemini-embedding-001
                  config  test_mrr_mean  test_mrr_sd
     weighted sum (beta)         0.9375       0.0322
production (w=0.85, k=5)         0.9297       0.0252
       RRF (w only, k=5)         0.9237       0.0323
           RRF (k and w)         0.9228       0.0294

--- cohere-embed-v4
                  config  test_mrr_mean  test_mrr_sd
     weighted sum (beta)         0.8766       0.0343
           RRF (k and w)         0.8501       0.0416
production (w=0.85, k=5)         0.8369       0.0357
       RRF (w only, k=5)         0.8355       0.0381



In [9]:
# Paired test: on each split, does tuned weighted-sum beat the production setting?
# Paired on the same test half, so split-to-split difficulty cancels out.
rng3 = np.random.default_rng(11)
print("per-split head-to-head vs the current production setting\n")
for m in MODELS:
    ws_mat, ws_lab = CONFIGS[m]["weighted sum (beta)"]
    rw_mat, rw_lab = CONFIGS[m]["RRF (w only, k=5)"]
    d_ws, d_rw, betas = [], [], []
    for _ in range(N_SPLIT):
        perm = rng3.permutation(len(ev))
        tr, te = perm[:len(ev) // 2], perm[len(ev) // 2:]
        base = BASE[m][te].mean()
        bw = int(np.argmax(ws_mat[:, tr].mean(axis=1)))
        br = int(np.argmax(rw_mat[:, tr].mean(axis=1)))
        d_ws.append(ws_mat[bw, te].mean() - base)
        d_rw.append(rw_mat[br, te].mean() - base)
        betas.append(ws_lab[bw])
    d_ws, d_rw, betas = np.array(d_ws), np.array(d_rw), np.array(betas)
    print(f"--- {m}")
    print(f"  weighted sum beats production in {100 * (d_ws > 0).mean():.0f}% of splits, "
          f"mean gain {d_ws.mean():+.4f} MRR")
    print(f"  tuned RRF    beats production in {100 * (d_rw > 0).mean():.0f}% of splits, "
          f"mean gain {d_rw.mean():+.4f} MRR")
    print(f"  beta chosen on the tune half: median {np.median(betas):.2f}, "
          f"90% interval [{np.percentile(betas, 5):.2f}, {np.percentile(betas, 95):.2f}]\n")

per-split head-to-head vs the current production setting

--- gemini-embedding-001
  weighted sum beats production in 71% of splits, mean gain +0.0084 MRR
  tuned RRF    beats production in 0% of splits, mean gain -0.0071 MRR
  beta chosen on the tune half: median 0.94, 90% interval [0.78, 0.94]

--- cohere-embed-v4
  weighted sum beats production in 98% of splits, mean gain +0.0409 MRR
  tuned RRF    beats production in 56% of splits, mean gain -0.0016 MRR
  beta chosen on the tune half: median 0.92, 90% interval [0.86, 0.92]



## 6. Can production actually implement weighted sum?

Sections 3 and 5 min–max normalise each ranker's scores across **all 531 articles**. Production cannot do that. `Retriever._semantic_candidates` asks Pinecone for the top `CANDIDATE_TOP_K` matches; Pinecone returns those and nothing else, so the full score vector does not exist at request time. Normalising over a 50-document candidate pool is a *different operation* from normalising over 531, and the difference is not cosmetic — min–max is defined by the extremes of whatever set it sees.

Recommending a method that cannot be built would be worse than recommending nothing, so the deployable variant is tested here on its own terms:

- take the top `CANDIDATE_TOP_K` from each ranker (exactly what production has),
- min–max normalise **within each pool**,
- give a document missing from a pool a score of 0 on that side,
- fuse with `beta`.

If this loses the advantage, the honest recommendation reverts to leaving fusion alone.

In [10]:
def wsum_candidate_pool(sem_scores, kw_scores, beta, pool=CANDIDATE_TOP_K):
    """Weighted sum using only what production can actually see at request time."""
    fused = np.zeros_like(sem_scores)
    for qi in range(sem_scores.shape[0]):
        s_idx = np.argpartition(-sem_scores[qi], pool)[:pool]
        k_idx = np.argpartition(-kw_scores[qi], pool)[:pool]
        s_v, k_v = sem_scores[qi, s_idx], kw_scores[qi, k_idx]
        s_n = (s_v - s_v.min()) / max(s_v.max() - s_v.min(), 1e-12)
        k_n = (k_v - k_v.min()) / max(k_v.max() - k_v.min(), 1e-12)
        row = np.zeros(sem_scores.shape[1])
        row[s_idx] += beta * s_n
        row[k_idx] += (1 - beta) * k_n
        fused[qi] = row
    return fused


rows = []
for m in MODELS:
    for b in BS[::2]:
        r = ranks_from_scores(wsum_candidate_pool(SIMS[m], BM_SCORES, b), PROD_TOP_K)
        rows.append({"model": m, "param": float(b), "recall@1": recall_at(r, 1), "mrr": mrr(r)})
pool_sweep = pd.DataFrame(rows)

print(f"weighted sum over the top-{CANDIDATE_TOP_K} candidate pools (deployable variant)")
for m in MODELS:
    s = pool_sweep[pool_sweep.model == m]
    b = s.loc[s.mrr.idxmax()]
    full = wsum_sweep[wsum_sweep.model == m]
    fb = full.loc[full.mrr.idxmax()]
    prod = at(rrf_sweep, m, RRF_W)
    print(f"\n  {m}")
    print(f"    pool-normalised best : beta={b.param:.2f}  recall@1 {b['recall@1']:.3f}  MRR {b.mrr:.4f}")
    print(f"    full-corpus best     : beta={fb.param:.2f}  recall@1 {fb['recall@1']:.3f}  MRR {fb.mrr:.4f}")
    print(f"    current production   : RRF w={RRF_W}                       MRR {prod:.4f}")

weighted sum over the top-50 candidate pools (deployable variant)

  gemini-embedding-001
    pool-normalised best : beta=0.86  recall@1 0.917  MRR 0.9464
    full-corpus best     : beta=0.94  recall@1 0.917  MRR 0.9472
    current production   : RRF w=0.85                       MRR 0.9306

  cohere-embed-v4
    pool-normalised best : beta=0.80  recall@1 0.800  MRR 0.8664
    full-corpus best     : beta=0.92  recall@1 0.817  MRR 0.8833
    current production   : RRF w=0.85                       MRR 0.8375


In [11]:
# Cross-validate the deployable variant the same way -- an in-sample peak proved
# misleading for RRF above, so it gets no benefit of the doubt here either.
rng4 = np.random.default_rng(23)
POOL_B = BS[::2]
for m in MODELS:
    mat = np.vstack([recip(ranks_from_scores(wsum_candidate_pool(SIMS[m], BM_SCORES, b), PROD_TOP_K))
                     for b in POOL_B])
    d, chosen = [], []
    for _ in range(300):
        perm = rng4.permutation(len(ev))
        tr, te = perm[:len(ev) // 2], perm[len(ev) // 2:]
        best = int(np.argmax(mat[:, tr].mean(axis=1)))
        d.append(mat[best, te].mean() - BASE[m][te].mean())
        chosen.append(POOL_B[best])
    d, chosen = np.array(d), np.array(chosen)
    print(f"{m}")
    print(f"  deployable weighted sum vs production: beats it in {100 * (d > 0).mean():.0f}% of splits, "
          f"mean gain {d.mean():+.4f} MRR")
    print(f"  beta chosen: median {np.median(chosen):.2f}, "
          f"90% interval [{np.percentile(chosen, 5):.2f}, {np.percentile(chosen, 95):.2f}]\n")

gemini-embedding-001
  deployable weighted sum vs production: beats it in 58% of splits, mean gain +0.0056 MRR
  beta chosen: median 0.86, 90% interval [0.70, 0.92]

cohere-embed-v4
  deployable weighted sum vs production: beats it in 81% of splits, mean gain +0.0134 MRR
  beta chosen: median 0.80, 90% interval [0.74, 0.88]



## Verdict

Two findings, and the first one is the opposite of what the sweep appeared to say.

### 1. Re-tuning the RRF weight is not worth doing — the apparent gain is overfitting

The in-sample sweep looked encouraging: Cohere's optimum sits at `w = 0.83` (MRR 0.850) against the production `0.85` (0.838), a gain of +0.0125. The bootstrap agreed the peak was stable — median optimum 0.82, tight 90% interval [0.75, 0.83], and `w = 0.85` was chosen in **0 of 2,000** resamples.

Cross-validation says that gain is not real:

| config | Gemini test MRR | Cohere test MRR |
|---|---|---|
| **production (w=0.85, k=5)** *(untuned)* | 0.9297 | 0.8369 |
| RRF (w tuned, k=5) | **0.9237** | 0.8355 |
| RRF (k and w tuned) | 0.9228 | 0.8501 |

Tuned RRF finishes **below the untuned production setting** for both models. Head-to-head per split, tuned RRF beats production in **0%** of splits for Gemini (mean −0.0071) and 56% for Cohere (mean −0.0016 — worse on average despite winning more often than not, i.e. it loses bigger than it wins).

This is worth dwelling on, because it is the trap Phase 6 fell into. A stable bootstrap optimum is **not** evidence that moving to it helps: the bootstrap only says "given these 60 questions, the argmax is reliably ~0.82". It cannot say the argmax generalises, because every resample is drawn from the same 60. Only holding data out reveals that the curve near the top is flat enough that chasing the peak buys nothing and costs variance.

**Recommendation: leave `RRF_W = 0.85` and `RRF_K = 5` alone.** Not because the current value is optimal — it is not — but because the difference across the whole plausible region is ~1.5–2 questions on 60, and picking a point in it from this eval set does not transfer.

### 2. Weighted sum genuinely beats RRF — but most of the gain is not deployable

Fusing normalised *scores* rather than ranks wins out-of-sample for both models, reversing Phase 6's conclusion:

| model | production (RRF) | weighted sum | beats production in |
|---|---|---|---|
| gemini | 0.9297 | **0.9375** | 71% of splits (+0.0084) |
| cohere | 0.8369 | **0.8766** | **98% of splits (+0.0409)** |

Cohere's result is the strong one — a large gain, consistent across nearly every split, with a stable `beta` (median 0.92, 90% interval [0.86, 0.92]).

**But that version cannot be built.** It min–max normalises across all 531 articles, and at request time Pinecone returns only the top `CANDIDATE_TOP_K` matches — the full score vector does not exist. Normalising within the 50-document candidate pool is what production could actually do, and it keeps only about a third of the benefit:

| model | deployable weighted sum vs production |
|---|---|
| gemini | 58% of splits, **+0.0056** — indistinguishable from a coin flip |
| cohere | 81% of splits, **+0.0134** — real but small |

### What to actually do

**If the project stays on Cohere:** switching fusion to pool-normalised weighted sum at `beta ≈ 0.80` is a genuine if modest win — +0.013 MRR out-of-sample, and in-sample recall@1 rises 0.750 → 0.800 (three questions). It requires `_semantic_candidates` to return Pinecone's scores rather than discarding them, which it currently does.

**If the project switches to Gemini:** leave fusion alone. 58% of splits is noise, and the change would add code for nothing.

### The thing worth taking away

Every fusion knob in this notebook is worth **at most ~0.013 MRR** out-of-sample, and the honest value for several of them is zero or negative. The still-undecided embedding-model choice is worth **~0.093** (Gemini 0.9297 vs Cohere 0.8369, both at the *current* production fusion).

Gemini with untuned, unmodified fusion (0.9297) beats Cohere with the best fusion this notebook could find, deployable or not (0.8766). **The model decision dominates every fusion decision by roughly seven to one**, so fusion work should wait behind it — and if Gemini is adopted, most of this becomes moot.

Nothing here has been changed in `src/retrieval.py`.
